# Transformer baseline for HaaS detection (vendor-disjoint folds)

Runs a fine-tuned transformer on **exactly the same vendor-disjoint folds** as the paper
(`results/logs/folds_vendor_disjoint.csv`, written by `analysis/classify_extra.py`) and writes
`results/logs/transformer.json`, which `paper/build_doc.py` reads if present.

* Runtime: about 10-15 min per seed for `distilroberta-base` on a free Colab T4 GPU.
* Nothing is tuned on test folds: the decision threshold is chosen on a vendor-disjoint validation
  split carved out of each training fold.
* Run three seeds and report the mean and the spread (the paper reports all seeds).

In [ ]:
!pip -q install "transformers>=4.44" "accelerate>=0.33" scikit-learn pandas

In [ ]:
# Upload folds_vendor_disjoint.csv (Colab: Files panel), or clone the repository:
# !git clone https://github.com/sagarkorde/Hacker_for_Hire && %cd Hacker_for_Hire
import os
CSV = "results/logs/folds_vendor_disjoint.csv" if os.path.exists("results/logs/folds_vendor_disjoint.csv") else "folds_vendor_disjoint.csv"
MODEL = "distilroberta-base"   # alternatives: "roberta-base", "bert-base-uncased"
SEEDS = [1, 2, 3]
MAX_LEN, EPOCHS, LR, BATCH = 128, 3, 2e-5, 32

In [ ]:
import json, numpy as np, pandas as pd, torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

d = pd.read_csv(CSV, keep_default_na=False)
assert d["fold"].nunique() == 5 and d["label"].sum() == 451, "unexpected fold file"
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(texts):
    return tok(list(texts), truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")

def predict_proba(model, texts):
    model.eval(); out = []
    for i in range(0, len(texts), 128):
        e = {k: v.to(dev) for k, v in encode(texts[i:i+128]).items()}
        with torch.no_grad():
            out.append(torch.softmax(model(**e).logits, -1)[:, 1].float().cpu().numpy())
    return np.concatenate(out)

def train(texts, y, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(dev)
    enc = encode(texts); y_t = torch.tensor(y)
    w = torch.tensor([1.0, (len(y) - y.sum()) / max(y.sum(), 1)], dtype=torch.float).to(dev)  # class weights
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(y) / BATCH))
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    lossf = torch.nn.CrossEntropyLoss(weight=w)
    for _ in range(EPOCHS):
        model.train(); perm = np.random.permutation(len(y))
        for i in range(0, len(y), BATCH):
            b = perm[i:i+BATCH]
            out = model(input_ids=enc["input_ids"][b].to(dev), attention_mask=enc["attention_mask"][b].to(dev))
            loss = lossf(out.logits, y_t[b].to(dev)); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step(); opt.zero_grad()
    return model

In [ ]:
def cluster_boot(y, pred, groups, other=None, n=2000, seed=0):
    rng = np.random.default_rng(seed); u = np.unique(groups)
    idx = {g: np.flatnonzero(groups == g) for g in u}; f, dd = [], []
    for _ in range(n):
        i = np.concatenate([idx[g] for g in rng.choice(u, len(u))])
        if y[i].sum() == 0: continue
        f.append(f1_score(y[i], pred[i]))
        if other is not None: dd.append(f1_score(y[i], pred[i]) - f1_score(y[i], other[i]))
    r = {"f1_ci95": [float(np.percentile(f, 2.5)), float(np.percentile(f, 97.5))]}
    if other is not None:
        r["diff_vs_rule"] = float(f1_score(y, pred) - f1_score(y, other))
        r["diff_vs_rule_ci95"] = [float(np.percentile(dd, 2.5)), float(np.percentile(dd, 97.5))]
    return r

results = {"model": MODEL, "max_len": MAX_LEN, "epochs": EPOCHS, "lr": LR, "batch": BATCH, "seeds": {}}
y_all, X_all, g_all = d["label"].to_numpy(), d["text"].to_numpy(), d["vendor"].to_numpy()
for seed in SEEDS:
    prob = np.zeros(len(d)); pred = np.zeros(len(d), dtype=int); fold_f1 = []
    for k in range(5):
        tr, te = np.flatnonzero(d["fold"] != k), np.flatnonzero(d["fold"] == k)
        # vendor-disjoint validation split inside the training folds, for the threshold only
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        a, v = next(gss.split(tr, y_all[tr], g_all[tr])); a, v = tr[a], tr[v]
        m = train(X_all[a], y_all[a], seed)
        pv = predict_proba(m, X_all[v])
        grid = np.linspace(0.05, 0.95, 19)
        thr = float(grid[np.argmax([f1_score(y_all[v], pv >= t) for t in grid])])
        m = train(X_all[tr], y_all[tr], seed)   # refit on the full training folds
        prob[te] = predict_proba(m, X_all[te]); pred[te] = (prob[te] >= thr).astype(int)
        fold_f1.append(float(f1_score(y_all[te], pred[te]))); print(seed, k, thr, fold_f1[-1])
        del m; torch.cuda.empty_cache()
    r = {"precision": float(precision_score(y_all, pred)), "recall": float(recall_score(y_all, pred)),
         "f1": float(f1_score(y_all, pred)), "ap": float(average_precision_score(y_all, prob)),
         "fold_f1": fold_f1, "fold_sd": float(np.std(fold_f1, ddof=1))}
    r.update(cluster_boot(y_all, pred, g_all, other=d["rule_pred"].to_numpy(), seed=seed))
    results["seeds"][str(seed)] = r; print(seed, r)

In [ ]:
f1s = [v["f1"] for v in results["seeds"].values()]
results["f1_mean"], results["f1_sd_over_seeds"] = float(np.mean(f1s)), float(np.std(f1s, ddof=1)) if len(f1s) > 1 else 0.0
os.makedirs("results/logs", exist_ok=True)
json.dump(results, open("results/logs/transformer.json", "w"), indent=2)
print(json.dumps({k: v for k, v in results.items() if k != "seeds"}, indent=1))
# Download results/logs/transformer.json and place it in the repository's results/logs/ before rebuilding the paper.